In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','orders','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
df = (
    spark.read.format('csv')
    .option('header','true')
    .option('inferSchema','true')
    .load('/Volumes/fmcg/bronze/source_fmcg/orders/landing')
    .withColumn('read_timestamp',F.current_timestamp())
    .select("*",'_metadata.file_name','_metadata.file_size')
)

df.display()

In [0]:
df.printSchema()
spark.table("fmcg.bronze.orders").printSchema()

In [0]:
df = df.withColumn("order_qty", F.col("order_qty").cast("double"))

df.write \
  .format("delta") \
  .option("delta.enableChangeDataFeed", "true") \
  .mode("append") \
  .saveAsTable("fmcg.bronze.orders")

In [0]:
df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('overwrite')\
    .saveAsTable('fmcg.bronze.stagging_orders')
df.count()

In [0]:
landing = '/Volumes/fmcg/bronze/source_fmcg/orders/landing/'
processed = '/Volumes/fmcg/bronze/source_fmcg/orders/processed/'
files = dbutils.fs.ls(landing)
for file_info in files: dbutils.fs.mv( file_info.path, f"{processed}/{file_info.name}", True )

In [0]:
df_orders = spark.sql('select * from fmcg.bronze.stagging_orders')
df_orders.display()

# **_# **_Keep only rows where order_qty is present_**_**

In [0]:
df_orders = df_orders.filter(F.col('order_qty').isNotNull())

# **_Clean customer_id  to keep numeric, else set to 9999999_**

In [0]:
df_orders = df_orders.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike("^[0-9]+$"),F.col('customer_id'))
    .otherwise('999999')
    .cast('string')
)

Remove weekday name from the date text

In [0]:
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'),r'^[A-Za-z]+,\s*',"")
)

# **Parse order_placement_date using multiple possible formats**

In [0]:
df_orders = df_orders.withColumn(
  'order_placement_date',
  F.coalesce(
    F.try_to_date('order_placement_date','yyyy/MM/dd'),
    F.try_to_date('order_placement_date','dd-MM-yyyy'),
    F.try_to_date('order_placement_date','dd/MM/yyyy'),
    F.try_to_date('order_placement_date','MMMM dd, yyyy',)
  )
)

Drop the duplicates

In [0]:
df_orders = df_orders.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])

In [0]:
df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).display()

In [0]:
df_products = spark.table('fmcg.silver.products')
df_joined =df_orders.join(df_products,on = 'product_id',how = 'inner').select(df_orders['*'],df_products['product_code'])
df_joined.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_table = 'fmcg.silver.orders'

# -----------------------------------
# 1. DEDUP SOURCE (CRITICAL FIX)
# -----------------------------------
window = Window.partitionBy(
    "order_id",
    "product_code",
    "customer_id",
    "order_placement_date"
).orderBy(F.col("read_timestamp").desc())

df_dedup = (
    df_joined
    .withColumn("rn", F.row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
)

# -----------------------------------
# 2. WRITE / MERGE
# -----------------------------------
if not spark.catalog.tableExists(silver_table):

    df_dedup.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(silver_table)

else:
    silver_delta = DeltaTable.forName(spark, silver_table)

    (
        silver_delta.alias("silver")
        .merge(
            df_dedup.alias("bronze"),
            """
            silver.order_id = bronze.order_id AND
            silver.product_code = bronze.product_code AND
            silver.customer_id = bronze.customer_id AND
            silver.order_placement_date = bronze.order_placement_date
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# **Stagging table to process just the arrived incremental data**

In [0]:
df_joined.write\
  .format('delta')\
    .option('delta.enableChangeDataFeed','true')\
      .mode('overwrite')\
        .saveAsTable('fmcg.silver.staging_orders')

Merge Data with parent company

In [0]:
df_child = spark.sql("SELECT order_placement_date as date FROM fmcg.silver.staging_orders")
df_child.display()

incremental_month_df = df_child.select(F.trunc("date", "MM").alias("start_month")).distinct() 
incremental_month_df.show() 
incremental_month_df.createOrReplaceTempView("incremental_months")

In [0]:
fact_df = spark.table("fmcg.gold.sb_fact_orders") \
    .withColumn("start_month", F.trunc(F.to_date("date"), "MM"))

monthly_table = fact_df.join(
    spark.table("incremental_months"),
    on="start_month",
    how="inner"
)
fact_df.show()

In [0]:
monthly_table.select('date').distinct().orderBy('date').show()

In [0]:
df_monthly_recalc = (
                    monthly_table.withColumn("month_start", F.trunc("date", "MM"))
                     .groupBy("month_start", "product_code", "customer_code")     
                     .agg(F.sum("sold_quantity").alias("sold_quantity"))     .withColumnRenamed("month_start", "date") ) 
                  

In [0]:
df_monthly_recalc.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, "fmcg.gold.fact_orders") 
gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()